# 06 (Block 4) — Gemini Live API: Native Bidirectional Audio Streaming
**Official Reference Pattern**: `google-gemini/cookbook/quickstarts/Get_started_LiveTranslate.ipynb`
**Antigravity IDE Usage**: Companion code walkthrough for the **Block 4 Live Translation Cloud Run Demo** (link shared live on the day).

## What to Observe
1. **Low-Latency Bidirectional Session (`client.aio.live.connect`)**: Streaming 16kHz input PCM audio frames and receiving 24kHz native audio output chunks over a single persistent WebSocket.
2. **Angular Client Equivalence**: In the Block 4 capstone, our Angular `AudioWorklet` + WebSockets/WebRTC client implements this exact send/receive loop reactively using **Angular Signals**.


In [2]:
from google import genai
from google.genai import types

client = genai.Client()
MODEL_ID = "gemini-3.8-flash"

live_config = types.LiveConnectConfig(
    response_modalities=[types.Modality.AUDIO],
    speech_config=types.SpeechConfig(
        voice_config=types.VoiceConfig(
            prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Aoede")
        )
    ),
    system_instruction=types.Content(
        parts=[types.Part.from_text(
            text="You are a real-time voice mission control assistant. Keep spoken responses concise."
        )]
    ),
)

async def stream_audio_turn(pcm_16khz_bytes: bytes):
    async with client.aio.live.connect(model=MODEL_ID, config=live_config) as session:
        await session.send_realtime_input(
            audio=types.Blob(data=pcm_16khz_bytes, mime_type="audio/pcm;rate=16000")
        )
        async for response in session.receive():
            if response.server_content and response.server_content.model_turn:
                for part in response.server_content.model_turn.parts:
                    if part.inline_data:
                        print(f"Received 24kHz PCM audio chunk: {len(part.inline_data.data)} bytes")

print("Configured Gemini Live bidirectional native audio session.")


Configured Gemini Live bidirectional native audio session.
